In [3]:
import pandas as pd
import numpy as np
df = pd.read_excel("veriseti.xlsx")

In [4]:
df.isnull().sum().sort_values(ascending=False)

tramer                     46832
ortalama_yakit_tuketimi    20688
takasa_uygun               17644
yakit_deposu               15045
motor_gucu                  1361
motor_hacmi                 1153
cekis                       1093
model                         71
vites_tipi                    12
arac_durumu                    6
kasa_tipi                      4
renk                           2
kilometre                      1
konum                          0
yakit_tipi                     0
yil                            0
seri                           0
marka                          0
ilan_tarihi                    0
boya_degisen                   0
fiyat                          0
kimden                         0
baslik                         0
dtype: int64

In [5]:
df["fiyat"] = df["fiyat"].str.replace(".", "")
df["fiyat"] = df["fiyat"].str.replace("TL", "")
df["fiyat"] = df["fiyat"].astype(int)

In [6]:
df["fiyat"].isnull().sum()

0

In [7]:
df = df[df["kilometre"] != ""]

In [8]:
df["kilometre"] = df["kilometre"].str.replace(".", "")
df["kilometre"] = df["kilometre"].str.replace(" km", "")


In [9]:
df["kilometre"].head(10)

0    124000
1     99000
2    207500
3    219000
4    335000
5     63000
6    249000
7    226000
8     18000
9    228000
Name: kilometre, dtype: object

In [10]:
df["kilometre"] = df["kilometre"].astype(float)

In [11]:
df["kilometre"].head(10)

0    124000.0
1     99000.0
2    207500.0
3    219000.0
4    335000.0
5     63000.0
6    249000.0
7    226000.0
8     18000.0
9    228000.0
Name: kilometre, dtype: float64

In [12]:
df = df[df["model"] != ""]

In [13]:
model_counts = df["model"].value_counts()

In [14]:
model_counts

model
1.6                               742
1.6 TDi BlueMotion Comfortline    732
1.3 Multijet Easy                 481
1.6 Comfort                       472
1.5 dCi Joy                       454
                                 ... 
A 190 Elegance                      1
i5 xDrive40 M Sport                 1
RX-8 1.3                            1
2.0 CRD Convertible                 1
200 Comfort                         1
Name: count, Length: 2951, dtype: int64

In [15]:
df["marka"] = df["marka"] + "_" + df["seri"]

In [16]:
df["marka"].value_counts()

marka
Opel_Astra           2686
Renault_Clio         2461
Renault_Megane       2243
Toyota_Corolla       2129
Volkswagen_Passat    2065
                     ... 
Hyundai_i20 N           1
Audi_E-Tron GT          1
Suzuki_Liana            1
Fiat_Croma              1
Chevrolet_Evanda        1
Name: count, Length: 426, dtype: int64

In [17]:
df["renk"].value_counts()

renk
Beyaz              19400
Gri                 8412
Siyah               5672
Gri (Gümüş)         3286
Kırmızı             2884
Mavi                2564
Füme                2149
Lacivert            1343
Bordo               1135
Gri (metalik)       1077
Yeşil                775
Mavi (metalik)       693
Bej                  609
Kahverengi           590
Gri (titanyum)       326
Şampanya             268
Yeşil (metalik)      262
Sarı                 197
Turkuaz              183
Diğer                179
Turuncu              119
Mor                   79
Altın                 47
Pembe                  5
Name: count, dtype: int64

In [18]:
df.dropna(subset=['renk'],inplace=True)

In [19]:
df["renk"].isnull().sum()

0

In [20]:
import re

df["renk"] = df["renk"].apply(lambda x: re.sub(r"\s*\(.*\)", "", x))

In [21]:
df["renk"].value_counts()

renk
Beyaz         19400
Gri           13101
Siyah          5672
Mavi           3257
Kırmızı        2884
Füme           2149
Lacivert       1343
Bordo          1135
Yeşil          1037
Bej             609
Kahverengi      590
Şampanya        268
Sarı            197
Turkuaz         183
Diğer           179
Turuncu         119
Mor              79
Altın            47
Pembe             5
Name: count, dtype: int64

In [22]:
renk_counts = df["renk"].value_counts()
rare_colors = renk_counts[renk_counts < 200].index

df["renk"] = df["renk"].replace(rare_colors, "Diger")

In [23]:
df["renk"].value_counts()

renk
Beyaz         19400
Gri           13101
Siyah          5672
Mavi           3257
Kırmızı        2884
Füme           2149
Lacivert       1343
Bordo          1135
Yeşil          1037
Diger           809
Bej             609
Kahverengi      590
Şampanya        268
Name: count, dtype: int64

In [24]:
import numpy as np

def temizle_motor_hacmi(x):
    if pd.isnull(x):
        return np.nan
    
    x = str(x).lower().replace("cc", "").replace("cm3", "").strip()
    
    if "-" in x:
        parts = x.split("-")
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except:
            return np.nan
    else:
        try:
            return float(x)
        except:
            return np.nan

df["motor_hacmi"] = df["motor_hacmi"].apply(temizle_motor_hacmi)

In [25]:
df["motor_hacmi"].describe()

count    49033.000000
mean      1502.312239
std        252.999962
min        796.000000
25%       1368.000000
50%       1500.500000
75%       1596.000000
max       6592.000000
Name: motor_hacmi, dtype: float64

In [26]:
df["motor_hacmi"] = (df["motor_hacmi"] / 100).round() * 100

In [27]:
import re

def extract_engine_from_text(text):
    if pd.isnull(text):
        return None
    
    match = re.search(r"(\d\.\d)", text)
    
    if match:
        return float(match.group(1)) * 1000
    return None

In [28]:
df["motor_hacmi"] = df.apply(
    lambda row: extract_engine_from_text(row["baslik"]) 
    if pd.isnull(row["motor_hacmi"]) else row["motor_hacmi"],
    axis=1
)

In [29]:
df["motor_hacmi"].value_counts()

motor_hacmi
1600.0    15329
1500.0    10925
1400.0     8683
1200.0     4915
1300.0     4651
2000.0     1996
1900.0     1147
1000.0     1039
1800.0      462
1100.0      459
1700.0      372
900.0       196
3000.0      140
2800.0      136
2100.0       89
2300.0       81
2500.0       38
0.0          30
3300.0       30
2200.0       30
5000.0       26
8000.0       24
9000.0       22
3800.0       21
3200.0       21
4000.0       19
7000.0       18
3500.0       18
6000.0       18
2700.0       18
2400.0       17
800.0        11
4800.0       11
4300.0       10
2600.0        9
2900.0        9
6500.0        4
5800.0        3
3600.0        3
7500.0        3
5300.0        3
9500.0        2
8500.0        2
4600.0        2
6600.0        2
4400.0        2
6100.0        1
9900.0        1
5100.0        1
5500.0        1
8600.0        1
4100.0        1
6800.0        1
6400.0        1
7700.0        1
9300.0        1
4500.0        1
8100.0        1
500.0         1
Name: count, dtype: int64

In [30]:
df["motor_hacmi"].isnull().sum()

1195

In [31]:
# önce marka bazlı doldur
df["motor_hacmi"] = df.groupby("marka")["motor_hacmi"].transform(
    lambda x: x.fillna(x.median())
)

In [32]:
df["motor_hacmi"].isnull().sum()

49

In [33]:
df["motor_gucu"].isnull().sum()

1360

In [34]:
df["motor_gucu"].value_counts()

motor_gucu
101 - 125 HP    4321
76 - 100 HP     3735
90 hp           3692
75 hp           2726
95 hp           2219
                ... 
630 hp             1
164 hp             1
59 hp              1
680 hp             1
279 hp             1
Name: count, Length: 203, dtype: int64

In [35]:
import numpy as np
import re

def temizle_motor_gucu(x):
    if pd.isnull(x):
        return np.nan
    
    x = str(x).lower().replace("hp", "").strip()
    
    if "-" in x:
        parts = x.split("-")
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except:
            return np.nan
    else:
        try:
            return float(x)
        except:
            return np.nan

df["motor_gucu"] = df["motor_gucu"].apply(temizle_motor_gucu)

In [36]:
df["motor_gucu"].describe()

count    50744.000000
mean       106.925627
std         37.054934
min         41.000000
25%         88.000000
50%        100.000000
75%        115.000000
max        761.000000
Name: motor_gucu, dtype: float64

In [37]:
df = df[(df["motor_gucu"] >= 50) & (df["motor_gucu"] <= 400)]

In [38]:
df["motor_gucu"] = df.groupby("marka")["motor_gucu"].transform(
    lambda x: x.fillna(x.median())
)

df["motor_gucu"] = df["motor_gucu"].fillna(df["motor_gucu"].median())

In [39]:
df["motor_gucu"].isnull().sum()

0

In [40]:
df.isnull().sum()

baslik                         0
konum                          0
fiyat                          0
ilan_tarihi                    0
marka                          0
seri                           0
model                         46
yil                            0
kilometre                      0
vites_tipi                     0
yakit_tipi                     0
kasa_tipi                      0
renk                           0
motor_hacmi                   18
motor_gucu                     0
cekis                         73
arac_durumu                    5
ortalama_yakit_tuketimi    19101
yakit_deposu               13696
boya_degisen                   0
takasa_uygun               16529
kimden                         0
tramer                     45256
dtype: int64

In [41]:
df.dropna(subset="model",inplace=True)

In [42]:
df.isnull().sum()

baslik                         0
konum                          0
fiyat                          0
ilan_tarihi                    0
marka                          0
seri                           0
model                          0
yil                            0
kilometre                      0
vites_tipi                     0
yakit_tipi                     0
kasa_tipi                      0
renk                           0
motor_hacmi                    9
motor_gucu                     0
cekis                         73
arac_durumu                    5
ortalama_yakit_tuketimi    19055
yakit_deposu               13675
boya_degisen                   0
takasa_uygun               16512
kimden                         0
tramer                     45210
dtype: int64

In [43]:
df["cekis"].value_counts()

cekis
Önden Çekiş         45162
Arkadan İtiş         4525
4WD (Sürekli)         801
-                      21
AWD (Elektronik)        4
Name: count, dtype: int64

In [44]:
df["cekis"] = df["cekis"].replace("-", pd.NA)

In [45]:
df["cekis"].value_counts()

cekis
Önden Çekiş         45162
Arkadan İtiş         4525
4WD (Sürekli)         801
AWD (Elektronik)        4
Name: count, dtype: int64

In [46]:
df["cekis"] = df["cekis"].fillna(df["cekis"].mode()[0])

In [47]:
df["cekis"] = df["cekis"].fillna(df["cekis"].mode()[0])

In [48]:
import re
import numpy as np

def parse_boya_detay(x):
    x = str(x).lower()
    
    boyali = 0
    degisen = 0
    
    # boyalı sayısı
    b = re.search(r"(\d+)\s*boyalı", x)
    if b:
        boyali = int(b.group(1))
    
    # değişen sayısı
    d = re.search(r"(\d+)\s*değişen", x)
    if d:
        degisen = int(d.group(1))
    
    # tamamen orijinal
    if "orjinal" in x:
        return 0, 0
    
    # belirtilmemiş
    if "belirtilmemiş" in x:
        return np.nan, np.nan
    
    return boyali, degisen

In [49]:
df[["boyali_sayisi", "degisen_sayisi"]] = df["boya_degisen"].apply(
    lambda x: pd.Series(parse_boya_detay(x))
)

In [50]:
df["hasar_bilinmiyor"] = df["boyali_sayisi"].isnull().astype(int)

df["boyali_sayisi"] = df["boyali_sayisi"].fillna(0)
df["degisen_sayisi"] = df["degisen_sayisi"].fillna(0)

In [51]:
df["hasar_bilinmiyor"].value_counts()

hasar_bilinmiyor
0    40669
1     9917
Name: count, dtype: int64

In [52]:
import pandas as pd
def tramer_kat(x):
    if pd.isna(x):  # x.isnull() yerine bunu kullan
        return "bilinmiyor"
    elif x == 0:
        return "yok"
    elif x < 5000:
        return "dusuk"
    elif x < 20000:
        return "orta"
    else:
        return "yuksek"

df["tramer_kategori"] = df["tramer"].apply(tramer_kat)

In [53]:
df["tramer"].isnull().sum()

45210

In [54]:
df["tramer"].isnull().sum()

45210

In [55]:
df["tramer"] = df["tramer"].fillna(0)

In [56]:
df["tramer_kategori"].value_counts()

tramer_kategori
bilinmiyor    45210
dusuk          3345
orta           1233
yuksek          748
yok              50
Name: count, dtype: int64

In [57]:
df["fiyat"].describe()

count    5.058600e+04
mean     7.573707e+05
std      2.172944e+06
min      1.000000e+04
25%      3.550000e+05
50%      5.850000e+05
75%      9.500000e+05
max      4.670000e+08
Name: fiyat, dtype: float64

In [58]:
df = df[(df["fiyat"] > 50000) & (df["fiyat"] < 3000000)]

In [59]:
df["fiyat"].describe()

count    4.996900e+04
mean     7.035791e+05
std      4.712570e+05
min      5.100000e+04
25%      3.500000e+05
50%      5.790000e+05
75%      9.270000e+05
max      2.999000e+06
Name: fiyat, dtype: float64

In [60]:
df = df[(df["kilometre"] >= 0) & (df["kilometre"] <= 500000)]

In [61]:
df["kilometre"].describe()

count     49694.000000
mean     200437.283877
std       94055.662516
min        6000.000000
25%      133000.000000
50%      200000.000000
75%      265000.000000
max      500000.000000
Name: kilometre, dtype: float64

In [62]:
df = df.reset_index(drop=True)

In [63]:
df["fiyat"].value_counts()

fiyat
350000     474
450000     450
550000     363
400000     348
250000     344
          ... 
968500       1
2241000      1
2749000      1
529750       1
510900       1
Name: count, Length: 3532, dtype: int64

In [64]:
df["sehir"] = df["konum"].apply(lambda x: str(x).split(",")[1].strip())

In [65]:
df["sehir"].value_counts()

sehir
Kütahya           977
Kahramanmaraş     973
Afyonkarahisar    970
Samsun            969
Gaziantep         963
                 ... 
Iğdır             121
Gümüşhane          89
Bayburt            66
Tunceli            36
Ardahan            30
Name: count, Length: 81, dtype: int64

In [66]:
df.drop(columns=["konum"], inplace=True)

In [67]:
df["ilan_yil"] = df["ilan_tarihi"].dt.year
df["ilan_ay"] = df["ilan_tarihi"].dt.month

In [68]:
df.drop(columns=["ilan_tarihi"], inplace=True)

In [73]:
df["kimden"].value_counts()

kimden
Sahibinden         27411
Galeriden          22035
Yetkili Bayiden      248
Name: count, dtype: int64

In [85]:
df["kimden"].isnull().sum()

0

In [96]:
df["ilan_yil"].isnull().sum()

0

In [97]:
df = df[df["tramer"] <= 500000]
df = df.reset_index(drop=True)

In [101]:
df.describe()

,fiyat,yil,kilometre,motor_hacmi,motor_gucu,tramer,boyali_sayisi,degisen_sayisi,hasar_bilinmiyor,ilan_yil,ilan_ay
count,4.960300e+04,49603.000000,49603.000000,49598.000000,49603.000000,49603.000000,49603.000000,49603.000000,49603.000000,49603.0,49603.000000
mean,7.050434e+05,2009.648993,200526.700321,1506.962982,104.841340,1059.533899,1.893978,0.435195,0.196682,2025.0,7.697478
std,4.712342e+05,8.345786,94056.674497,391.332820,29.295546,9503.657887,2.914447,0.842292,0.397494,0.0,0.583712
min,5.100000e+04,1973.000000,6000.000000,0.000000,50.000000,0.000000,0.000000,0.000000,0.000000,2025.0,1.000000
25%,3.550000e+05,2004.000000,133000.000000,1400.000000,88.000000,0.000000,0.000000,0.000000,0.000000,2025.0,7.000000
50%,5.800000e+05,2011.000000,200000.000000,1500.000000,100.000000,0.000000,0.000000,0.000000,0.000000,2025.0,8.000000
75%,9.300000e+05,2016.000000,265000.000000,1600.000000,115.000000,0.000000,3.000000,1.000000,0.000000,2025.0,8.000000
max,2.999000e+06,2025.000000,500000.000000,9900.000000,388.000000,500000.000000,12.000000,11.000000,1.000000,2025.0,8.000000


In [109]:
df["tramer_kategori"].value_counts()

tramer_kategori
bilinmiyor    44387
dusuk          3320
orta           1219
yuksek          628
yok              49
Name: count, dtype: int64

In [115]:
df["motor_hacmi"].value_counts()

motor_hacmi
1600.0    14673
1500.0    10646
1400.0     8750
1200.0     4915
1300.0     4568
2000.0     1894
1000.0     1038
1900.0      924
1100.0      468
1800.0      445
1700.0      344
900.0       188
3000.0      101
2800.0       89
2100.0       87
2300.0       69
2500.0       33
0.0          30
2200.0       29
6000.0       26
3300.0       25
5000.0       24
8000.0       24
4500.0       23
9000.0       21
3200.0       20
3500.0       18
2400.0       17
2700.0       17
7000.0       15
4000.0       14
2600.0        9
2900.0        6
8500.0        5
6500.0        4
4300.0        4
3800.0        4
800.0         3
4800.0        3
1250.0        3
7500.0        3
5300.0        2
3600.0        2
5800.0        2
4600.0        2
9500.0        2
9300.0        1
6400.0        1
4400.0        1
6100.0        1
8600.0        1
5500.0        1
8100.0        1
9900.0        1
500.0         1
Name: count, dtype: int64

In [122]:
df = df[(df["motor_hacmi"] >= 800) & (df["motor_hacmi"] <= 3000)]

In [123]:
df.describe()

,fiyat,yil,kilometre,motor_hacmi,motor_gucu,tramer,boyali_sayisi,degisen_sayisi,hasar_bilinmiyor,ilan_yil,ilan_ay
count,4.931600e+04,49316.000000,49316.000000,49316.000000,49316.000000,49316.000000,49316.000000,49316.000000,49316.000000,49316.0,49316.000000
mean,7.022452e+05,2009.620407,200982.663963,1487.491889,104.541670,1058.854449,1.898086,0.436086,0.197380,2025.0,7.696468
std,4.694233e+05,8.325118,93676.403665,224.780442,28.436133,9521.351473,2.915470,0.842517,0.398026,0.0,0.583900
min,5.100000e+04,1973.000000,6000.000000,800.000000,50.000000,0.000000,0.000000,0.000000,0.000000,2025.0,1.000000
25%,3.523750e+05,2004.000000,134000.000000,1400.000000,88.000000,0.000000,0.000000,0.000000,0.000000,2025.0,7.000000
50%,5.750000e+05,2011.000000,201000.000000,1500.000000,100.000000,0.000000,0.000000,0.000000,0.000000,2025.0,8.000000
75%,9.250000e+05,2016.000000,265000.000000,1600.000000,115.000000,0.000000,3.000000,1.000000,0.000000,2025.0,8.000000
max,2.999000e+06,2025.000000,500000.000000,3000.000000,363.000000,500000.000000,12.000000,11.000000,1.000000,2025.0,8.000000


In [124]:
df["motor_hacmi"].value_counts()

motor_hacmi
1600.0    14673
1500.0    10646
1400.0     8750
1200.0     4915
1300.0     4568
2000.0     1894
1000.0     1038
1900.0      924
1100.0      468
1800.0      445
1700.0      344
900.0       188
3000.0      101
2800.0       89
2100.0       87
2300.0       69
2500.0       33
2200.0       29
2400.0       17
2700.0       17
2600.0        9
2900.0        6
800.0         3
1250.0        3
Name: count, dtype: int64

In [125]:
df.isnull().sum()

baslik                         0
fiyat                          0
marka                          0
seri                           0
model                          0
yil                            0
kilometre                      0
vites_tipi                     0
yakit_tipi                     0
kasa_tipi                      0
renk                           0
motor_hacmi                    0
motor_gucu                     0
cekis                          0
arac_durumu                    5
ortalama_yakit_tuketimi    18142
yakit_deposu               12830
boya_degisen                   0
takasa_uygun               16335
kimden                         0
tramer                         0
boyali_sayisi                  0
degisen_sayisi                 0
hasar_bilinmiyor               0
tramer_kategori                0
sehir                          0
ilan_yil                       0
ilan_ay                        0
dtype: int64

In [126]:
df.drop(columns=["ortalama_yakit_tuketimi","yakit_deposu","takasa_uygun"],inplace=True)

In [127]:
df.isnull().sum()

baslik              0
fiyat               0
marka               0
seri                0
model               0
yil                 0
kilometre           0
vites_tipi          0
yakit_tipi          0
kasa_tipi           0
renk                0
motor_hacmi         0
motor_gucu          0
cekis               0
arac_durumu         5
boya_degisen        0
kimden              0
tramer              0
boyali_sayisi       0
degisen_sayisi      0
hasar_bilinmiyor    0
tramer_kategori     0
sehir               0
ilan_yil            0
ilan_ay             0
dtype: int64

In [128]:
df.drop("arac_durumu",axis=1,inplace=True)

In [129]:
df.drop("seri",axis=1,inplace=True)

In [130]:
df.isnull().sum()

baslik              0
fiyat               0
marka               0
model               0
yil                 0
kilometre           0
vites_tipi          0
yakit_tipi          0
kasa_tipi           0
renk                0
motor_hacmi         0
motor_gucu          0
cekis               0
boya_degisen        0
kimden              0
tramer              0
boyali_sayisi       0
degisen_sayisi      0
hasar_bilinmiyor    0
tramer_kategori     0
sehir               0
ilan_yil            0
ilan_ay             0
dtype: int64

In [131]:
df2=df.copy()

In [132]:
marka_counts = df["marka"].value_counts()
rare_brands = marka_counts[marka_counts < 20].index
df["marka"] = df["marka"].apply(
    lambda x: "diger" if x in rare_brands else x
)

In [133]:
df["marka"].value_counts()

marka
Opel_Astra           2619
Renault_Clio         2366
Renault_Megane       2198
Toyota_Corolla       2041
Volkswagen_Passat    1996
                     ... 
Mitsubishi_Colt        23
Citroen_Saxo           23
Volvo_S80              22
Renault_Modus          21
Ford_B-Max             20
Name: count, Length: 155, dtype: int64

In [134]:
df2=df.copy()

In [135]:
df["yas"] = 2025 - df["yil"]
df.drop(columns=["yil"], inplace=True)

In [136]:
df["fiyat"].describe()

count    4.931600e+04
mean     7.022452e+05
std      4.694233e+05
min      5.100000e+04
25%      3.523750e+05
50%      5.750000e+05
75%      9.250000e+05
max      2.999000e+06
Name: fiyat, dtype: float64

In [137]:
df.describe()

,fiyat,kilometre,motor_hacmi,motor_gucu,tramer,boyali_sayisi,degisen_sayisi,hasar_bilinmiyor,ilan_yil,ilan_ay,yas
count,4.931600e+04,49316.000000,49316.000000,49316.000000,49316.000000,49316.000000,49316.000000,49316.000000,49316.0,49316.000000,49316.000000
mean,7.022452e+05,200982.663963,1487.491889,104.541670,1058.854449,1.898086,0.436086,0.197380,2025.0,7.696468,15.379593
std,4.694233e+05,93676.403665,224.780442,28.436133,9521.351473,2.915470,0.842517,0.398026,0.0,0.583900,8.325118
min,5.100000e+04,6000.000000,800.000000,50.000000,0.000000,0.000000,0.000000,0.000000,2025.0,1.000000,0.000000
25%,3.523750e+05,134000.000000,1400.000000,88.000000,0.000000,0.000000,0.000000,0.000000,2025.0,7.000000,9.000000
50%,5.750000e+05,201000.000000,1500.000000,100.000000,0.000000,0.000000,0.000000,0.000000,2025.0,8.000000,14.000000
75%,9.250000e+05,265000.000000,1600.000000,115.000000,0.000000,3.000000,1.000000,0.000000,2025.0,8.000000,21.000000
max,2.999000e+06,500000.000000,3000.000000,363.000000,500000.000000,12.000000,11.000000,1.000000,2025.0,8.000000,52.000000


In [138]:
df.drop("boya_degisen",axis=1,inplace=True)

In [139]:
df.columns

Index(['baslik', 'fiyat', 'marka', 'model', 'kilometre', 'vites_tipi',
       'yakit_tipi', 'kasa_tipi', 'renk', 'motor_hacmi', 'motor_gucu', 'cekis',
       'kimden', 'tramer', 'boyali_sayisi', 'degisen_sayisi',
       'hasar_bilinmiyor', 'tramer_kategori', 'sehir', 'ilan_yil', 'ilan_ay',
       'yas'],
      dtype='object')

In [140]:
df

,baslik,fiyat,marka,model,kilometre,vites_tipi,yakit_tipi,kasa_tipi,renk,motor_hacmi,...,kimden,tramer,boyali_sayisi,degisen_sayisi,hasar_bilinmiyor,tramer_kategori,sehir,ilan_yil,ilan_ay,yas
0,2022 MEGANE 1.3 TCE 140 HP JOY COMFORT EDC SER...,1169000,Renault_Megane,1.3 TCe Joy Comfort,124000.0,Otomatik,Benzin,Sedan,Beyaz,1300.0,...,Galeriden,71300.0,0.0,1.0,0,yuksek,Adana,2025,8,3
1,2022 SKODA OCTAVİA 1.0 E-TEC ELİTE DSG SERVİS ...,1520000,Skoda_Octavia,1.0 e-Tec Elite,99000.0,Otomatik,Hibrit,Sedan,Gri,1300.0,...,Galeriden,56060.0,1.0,0.0,0,yuksek,Adana,2025,8,3
2,Sahibinden HATASIZ BOYASIZ Peugeot 207,469999,Peugeot_207,1.4 Trendy,207500.0,Düz,LPG & Benzin,Hatchback/5,Gri,1400.0,...,Sahibinden,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,16
3,BEY CLASS'TAN AUDİ A4 QUATRO S-TRONİC 177 HP D...,1150000,Audi_A4,A4 Sedan 2.0 TDI Quattro,219000.0,Otomatik,Dizel,Sedan,Beyaz,1900.0,...,Galeriden,0.0,4.0,1.0,0,bilinmiyor,Adana,2025,8,12
4,BEY CLASS'TAN BMW 520İ/LONG/HAYALET//NBT/K ISI...,1359000,BMW_5 Serisi,520Li,335000.0,Otomatik,Benzin,Sedan,Beyaz,1900.0,...,Galeriden,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49598,ÖZER OTOMOTİVDEN 1994 MERCEDES C180 SANRUFLU K...,365000,Mercedes - Benz_C,C 180 Elegance,330000.0,Otomatik,LPG & Benzin,Sedan,Mavi,1700.0,...,Galeriden,0.0,11.0,0.0,0,bilinmiyor,Düzce,2025,3,31
49599,ÖZKAN OTOMOTİV DEN 1993 CONCORT 2.0 LPG LI,177500,Renault_R 21,2.0 Concorde,324000.0,Düz,Benzin,Sedan,Füme,2000.0,...,Galeriden,0.0,0.0,0.0,1,bilinmiyor,Düzce,2025,3,32
49600,ÖZKAN OTOMOTİV DEN 2011 CLIO 1.4 LPG LI,357500,Renault_Symbol,1.4 Authentique,332000.0,Düz,Benzin,Sedan,Gri,1400.0,...,Galeriden,0.0,0.0,0.0,1,bilinmiyor,Düzce,2025,2,14
49601,ÖZKAN OTOMOTİV DEN 1987 RENULT BAKIMLI MASRAFSIZ,125000,Renault_R 12,TX,178000.0,Düz,Benzin,Sedan,Lacivert,1400.0,...,Galeriden,0.0,0.0,0.0,1,bilinmiyor,Düzce,2025,2,39


In [141]:
df["marka"].describe()

count          49316
unique           155
top       Opel_Astra
freq            2619
Name: marka, dtype: object

In [142]:
df=df[df["marka"]!="diger"]

In [143]:
df.describe()

,fiyat,kilometre,motor_hacmi,motor_gucu,tramer,boyali_sayisi,degisen_sayisi,hasar_bilinmiyor,ilan_yil,ilan_ay,yas
count,4.825800e+04,48258.000000,48258.000000,48258.000000,48258.000000,48258.000000,48258.000000,48258.000000,48258.0,48258.000000,48258.000000
mean,7.026173e+05,200657.583779,1483.170666,104.148452,1049.796946,1.898276,0.439513,0.196133,2025.0,7.697107,15.312881
std,4.655721e+05,93457.440863,213.540612,27.630413,9481.127193,2.909916,0.845505,0.397075,0.0,0.582744,8.299793
min,5.100000e+04,6000.000000,900.000000,50.000000,0.000000,0.000000,0.000000,0.000000,2025.0,1.000000,0.000000
25%,3.550000e+05,133368.500000,1400.000000,88.000000,0.000000,0.000000,0.000000,0.000000,2025.0,7.000000,9.000000
50%,5.800000e+05,200000.000000,1500.000000,100.000000,0.000000,0.000000,0.000000,0.000000,2025.0,8.000000,14.000000
75%,9.250000e+05,265000.000000,1600.000000,115.000000,0.000000,3.000000,1.000000,0.000000,2025.0,8.000000,21.000000
max,2.999000e+06,500000.000000,3000.000000,363.000000,500000.000000,12.000000,11.000000,1.000000,2025.0,8.000000,52.000000


In [146]:
df.head()

,baslik,fiyat,marka,model,kilometre,vites_tipi,yakit_tipi,kasa_tipi,renk,motor_hacmi,...,kimden,tramer,boyali_sayisi,degisen_sayisi,hasar_bilinmiyor,tramer_kategori,sehir,ilan_yil,ilan_ay,yas
0,2022 MEGANE 1.3 TCE 140 HP JOY COMFORT EDC SER...,1169000,Renault_Megane,1.3 TCe Joy Comfort,124000.0,Otomatik,Benzin,Sedan,Beyaz,1300.0,...,Galeriden,71300.0,0.0,1.0,0,yuksek,Adana,2025,8,3
1,2022 SKODA OCTAVİA 1.0 E-TEC ELİTE DSG SERVİS ...,1520000,Skoda_Octavia,1.0 e-Tec Elite,99000.0,Otomatik,Hibrit,Sedan,Gri,1300.0,...,Galeriden,56060.0,1.0,0.0,0,yuksek,Adana,2025,8,3
2,Sahibinden HATASIZ BOYASIZ Peugeot 207,469999,Peugeot_207,1.4 Trendy,207500.0,Düz,LPG & Benzin,Hatchback/5,Gri,1400.0,...,Sahibinden,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,16
3,BEY CLASS'TAN AUDİ A4 QUATRO S-TRONİC 177 HP D...,1150000,Audi_A4,A4 Sedan 2.0 TDI Quattro,219000.0,Otomatik,Dizel,Sedan,Beyaz,1900.0,...,Galeriden,0.0,4.0,1.0,0,bilinmiyor,Adana,2025,8,12
4,BEY CLASS'TAN BMW 520İ/LONG/HAYALET//NBT/K ISI...,1359000,BMW_5 Serisi,520Li,335000.0,Otomatik,Benzin,Sedan,Beyaz,1900.0,...,Galeriden,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,12


In [165]:
df.drop(columns="baslik",inplace=True)

C:\Users\Yasin\AppData\Local\Temp\ipykernel_3772\3175257344.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns="baslik",inplace=True)


In [168]:
df.describe()

,fiyat,kilometre,motor_hacmi,motor_gucu,tramer,boyali_sayisi,degisen_sayisi,hasar_bilinmiyor,ilan_yil,ilan_ay,yas
count,4.825800e+04,48258.000000,48258.000000,48258.000000,48258.000000,48258.000000,48258.000000,48258.000000,48258.0,48258.000000,48258.000000
mean,7.026173e+05,200657.583779,1483.170666,104.148452,1049.796946,1.898276,0.439513,0.196133,2025.0,7.697107,15.312881
std,4.655721e+05,93457.440863,213.540612,27.630413,9481.127193,2.909916,0.845505,0.397075,0.0,0.582744,8.299793
min,5.100000e+04,6000.000000,900.000000,50.000000,0.000000,0.000000,0.000000,0.000000,2025.0,1.000000,0.000000
25%,3.550000e+05,133368.500000,1400.000000,88.000000,0.000000,0.000000,0.000000,0.000000,2025.0,7.000000,9.000000
50%,5.800000e+05,200000.000000,1500.000000,100.000000,0.000000,0.000000,0.000000,0.000000,2025.0,8.000000,14.000000
75%,9.250000e+05,265000.000000,1600.000000,115.000000,0.000000,3.000000,1.000000,0.000000,2025.0,8.000000,21.000000
max,2.999000e+06,500000.000000,3000.000000,363.000000,500000.000000,12.000000,11.000000,1.000000,2025.0,8.000000,52.000000


In [172]:
df.head()

,fiyat,marka,model,kilometre,vites_tipi,yakit_tipi,kasa_tipi,renk,motor_hacmi,motor_gucu,...,kimden,tramer,boyali_sayisi,degisen_sayisi,hasar_bilinmiyor,tramer_kategori,sehir,ilan_yil,ilan_ay,yas
0,1169000,Renault_Megane,1.3 TCe Joy Comfort,124000.0,Otomatik,Benzin,Sedan,Beyaz,1300.0,138.0,...,Galeriden,71300.0,0.0,1.0,0,yuksek,Adana,2025,8,3
1,1520000,Skoda_Octavia,1.0 e-Tec Elite,99000.0,Otomatik,Hibrit,Sedan,Gri,1300.0,113.0,...,Galeriden,56060.0,1.0,0.0,0,yuksek,Adana,2025,8,3
2,469999,Peugeot_207,1.4 Trendy,207500.0,Düz,LPG & Benzin,Hatchback/5,Gri,1400.0,91.0,...,Sahibinden,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,16
3,1150000,Audi_A4,A4 Sedan 2.0 TDI Quattro,219000.0,Otomatik,Dizel,Sedan,Beyaz,1900.0,188.0,...,Galeriden,0.0,4.0,1.0,0,bilinmiyor,Adana,2025,8,12
4,1359000,BMW_5 Serisi,520Li,335000.0,Otomatik,Benzin,Sedan,Beyaz,1900.0,188.0,...,Galeriden,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,12


In [175]:
df=df.copy()
df["fiyat2"]=df["fiyat"]

In [203]:
df.head()

,fiyat,marka,model,kilometre,vites_tipi,yakit_tipi,kasa_tipi,renk,motor_hacmi,motor_gucu,...,tramer,boyali_sayisi,degisen_sayisi,hasar_bilinmiyor,tramer_kategori,sehir,ilan_yil,ilan_ay,yas,fiyat2
0,1169000,Renault_Megane,1.3 TCe Joy Comfort,124000.0,Otomatik,Benzin,Sedan,Beyaz,1300.0,138.0,...,71300.0,0.0,1.0,0,yuksek,Adana,2025,8,3,1169000
1,1520000,Skoda_Octavia,1.0 e-Tec Elite,99000.0,Otomatik,Hibrit,Sedan,Gri,1300.0,113.0,...,56060.0,1.0,0.0,0,yuksek,Adana,2025,8,3,1520000
2,469999,Peugeot_207,1.4 Trendy,207500.0,Düz,LPG & Benzin,Hatchback/5,Gri,1400.0,91.0,...,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,16,469999
3,1150000,Audi_A4,A4 Sedan 2.0 TDI Quattro,219000.0,Otomatik,Dizel,Sedan,Beyaz,1900.0,188.0,...,0.0,4.0,1.0,0,bilinmiyor,Adana,2025,8,12,1150000
4,1359000,BMW_5 Serisi,520Li,335000.0,Otomatik,Benzin,Sedan,Beyaz,1900.0,188.0,...,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,12,1359000


In [205]:
df.drop(columns="fiyat",inplace=True)

In [207]:
df.head()

,marka,model,kilometre,vites_tipi,yakit_tipi,kasa_tipi,renk,motor_hacmi,motor_gucu,cekis,...,tramer,boyali_sayisi,degisen_sayisi,hasar_bilinmiyor,tramer_kategori,sehir,ilan_yil,ilan_ay,yas,fiyat2
0,Renault_Megane,1.3 TCe Joy Comfort,124000.0,Otomatik,Benzin,Sedan,Beyaz,1300.0,138.0,Önden Çekiş,...,71300.0,0.0,1.0,0,yuksek,Adana,2025,8,3,1169000
1,Skoda_Octavia,1.0 e-Tec Elite,99000.0,Otomatik,Hibrit,Sedan,Gri,1300.0,113.0,Önden Çekiş,...,56060.0,1.0,0.0,0,yuksek,Adana,2025,8,3,1520000
2,Peugeot_207,1.4 Trendy,207500.0,Düz,LPG & Benzin,Hatchback/5,Gri,1400.0,91.0,Önden Çekiş,...,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,16,469999
3,Audi_A4,A4 Sedan 2.0 TDI Quattro,219000.0,Otomatik,Dizel,Sedan,Beyaz,1900.0,188.0,4WD (Sürekli),...,0.0,4.0,1.0,0,bilinmiyor,Adana,2025,8,12,1150000
4,BMW_5 Serisi,520Li,335000.0,Otomatik,Benzin,Sedan,Beyaz,1900.0,188.0,Arkadan İtiş,...,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,12,1359000


In [209]:
df.rename(columns={"fiyat2":"fiyat"},inplace=True)

In [211]:
df.rename(columns={"hasar_bilinmiyor":"tramer_bilinmiyor"},inplace=True)

In [213]:
df.head()

,marka,model,kilometre,vites_tipi,yakit_tipi,kasa_tipi,renk,motor_hacmi,motor_gucu,cekis,...,tramer,boyali_sayisi,degisen_sayisi,tramer_bilinmiyor,tramer_kategori,sehir,ilan_yil,ilan_ay,yas,fiyat
0,Renault_Megane,1.3 TCe Joy Comfort,124000.0,Otomatik,Benzin,Sedan,Beyaz,1300.0,138.0,Önden Çekiş,...,71300.0,0.0,1.0,0,yuksek,Adana,2025,8,3,1169000
1,Skoda_Octavia,1.0 e-Tec Elite,99000.0,Otomatik,Hibrit,Sedan,Gri,1300.0,113.0,Önden Çekiş,...,56060.0,1.0,0.0,0,yuksek,Adana,2025,8,3,1520000
2,Peugeot_207,1.4 Trendy,207500.0,Düz,LPG & Benzin,Hatchback/5,Gri,1400.0,91.0,Önden Çekiş,...,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,16,469999
3,Audi_A4,A4 Sedan 2.0 TDI Quattro,219000.0,Otomatik,Dizel,Sedan,Beyaz,1900.0,188.0,4WD (Sürekli),...,0.0,4.0,1.0,0,bilinmiyor,Adana,2025,8,12,1150000
4,BMW_5 Serisi,520Li,335000.0,Otomatik,Benzin,Sedan,Beyaz,1900.0,188.0,Arkadan İtiş,...,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,12,1359000


In [215]:
model_counts = df['model'].value_counts()
print(model_counts.head(10))   # en sık olanlar
print(model_counts.tail(3000))   # en nadir olanlar

model
1.6 TDi BlueMotion Comfortline    721
1.3 Multijet Easy                 474
1.6 Comfort                       466
1.5 dCi Joy                       446
1.6                               443
1.5 dCi Touch                     431
1.6 TDi Comfortline               330
1.5 dCi Authentique               327
1.5 dCi Icon                      323
1.6 SX                            321
Name: count, dtype: int64
model
1.6 TDi BlueMotion Comfortline    721
1.3 Multijet Easy                 474
1.6 Comfort                       466
1.5 dCi Joy                       446
1.6                               443
                                 ... 
1.5 T3 R-Design                     1
1.1 Hiper                           1
2.4                                 1
1.6 D-4D Premium Navi               1
1.25 CVVT Elegance Tekno            1
Name: count, Length: 2455, dtype: int64


In [217]:
rare_models = model_counts[model_counts < 13]
print(f"Nadir model sayısı: {len(rare_models)}")

Nadir model sayısı: 1737


In [219]:
df['model'] = df['model'].apply(
    lambda x: x if x not in rare_models.index else 'diger'
)

In [221]:
df["model"].value_counts()

model
diger                              6266
1.6 TDi BlueMotion Comfortline      721
1.3 Multijet Easy                   474
1.6 Comfort                         466
1.5 dCi Joy                         446
                                   ... 
A 180 d Style                        13
1.6 CDTI Grand Sport Excellence      13
1.5 Blue DCI Joy Comfort             13
1.2 GL                               13
1.4 HDi Urban Move                   13
Name: count, Length: 719, dtype: int64

In [223]:
df.describe()

,kilometre,motor_hacmi,motor_gucu,tramer,boyali_sayisi,degisen_sayisi,tramer_bilinmiyor,ilan_yil,ilan_ay,yas,fiyat
count,48258.000000,48258.000000,48258.000000,48258.000000,48258.000000,48258.000000,48258.000000,48258.0,48258.000000,48258.000000,4.825800e+04
mean,200657.583779,1483.170666,104.148452,1049.796946,1.898276,0.439513,0.196133,2025.0,7.697107,15.312881,7.026173e+05
std,93457.440863,213.540612,27.630413,9481.127193,2.909916,0.845505,0.397075,0.0,0.582744,8.299793,4.655721e+05
min,6000.000000,900.000000,50.000000,0.000000,0.000000,0.000000,0.000000,2025.0,1.000000,0.000000,5.100000e+04
25%,133368.500000,1400.000000,88.000000,0.000000,0.000000,0.000000,0.000000,2025.0,7.000000,9.000000,3.550000e+05
50%,200000.000000,1500.000000,100.000000,0.000000,0.000000,0.000000,0.000000,2025.0,8.000000,14.000000,5.800000e+05
75%,265000.000000,1600.000000,115.000000,0.000000,3.000000,1.000000,0.000000,2025.0,8.000000,21.000000,9.250000e+05
max,500000.000000,3000.000000,363.000000,500000.000000,12.000000,11.000000,1.000000,2025.0,8.000000,52.000000,2.999000e+06


In [225]:
df.head()

,marka,model,kilometre,vites_tipi,yakit_tipi,kasa_tipi,renk,motor_hacmi,motor_gucu,cekis,...,tramer,boyali_sayisi,degisen_sayisi,tramer_bilinmiyor,tramer_kategori,sehir,ilan_yil,ilan_ay,yas,fiyat
0,Renault_Megane,1.3 TCe Joy Comfort,124000.0,Otomatik,Benzin,Sedan,Beyaz,1300.0,138.0,Önden Çekiş,...,71300.0,0.0,1.0,0,yuksek,Adana,2025,8,3,1169000
1,Skoda_Octavia,1.0 e-Tec Elite,99000.0,Otomatik,Hibrit,Sedan,Gri,1300.0,113.0,Önden Çekiş,...,56060.0,1.0,0.0,0,yuksek,Adana,2025,8,3,1520000
2,Peugeot_207,1.4 Trendy,207500.0,Düz,LPG & Benzin,Hatchback/5,Gri,1400.0,91.0,Önden Çekiş,...,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,16,469999
3,Audi_A4,A4 Sedan 2.0 TDI Quattro,219000.0,Otomatik,Dizel,Sedan,Beyaz,1900.0,188.0,4WD (Sürekli),...,0.0,4.0,1.0,0,bilinmiyor,Adana,2025,8,12,1150000
4,BMW_5 Serisi,diger,335000.0,Otomatik,Benzin,Sedan,Beyaz,1900.0,188.0,Arkadan İtiş,...,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,12,1359000


In [229]:
df.head()

,marka,model,kilometre,vites_tipi,yakit_tipi,kasa_tipi,renk,motor_hacmi,motor_gucu,cekis,...,tramer,boyali_sayisi,degisen_sayisi,tramer_bilinmiyor,tramer_kategori,sehir,ilan_yil,ilan_ay,yas,fiyat
0,Renault_Megane_1.3 TCe Joy Comfort,1.3 TCe Joy Comfort,124000.0,Otomatik,Benzin,Sedan,Beyaz,1300.0,138.0,Önden Çekiş,...,71300.0,0.0,1.0,0,yuksek,Adana,2025,8,3,1169000
1,Skoda_Octavia_1.0 e-Tec Elite,1.0 e-Tec Elite,99000.0,Otomatik,Hibrit,Sedan,Gri,1300.0,113.0,Önden Çekiş,...,56060.0,1.0,0.0,0,yuksek,Adana,2025,8,3,1520000
2,Peugeot_207_1.4 Trendy,1.4 Trendy,207500.0,Düz,LPG & Benzin,Hatchback/5,Gri,1400.0,91.0,Önden Çekiş,...,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,16,469999
3,Audi_A4_A4 Sedan 2.0 TDI Quattro,A4 Sedan 2.0 TDI Quattro,219000.0,Otomatik,Dizel,Sedan,Beyaz,1900.0,188.0,4WD (Sürekli),...,0.0,4.0,1.0,0,bilinmiyor,Adana,2025,8,12,1150000
4,BMW_5 Serisi_diger,diger,335000.0,Otomatik,Benzin,Sedan,Beyaz,1900.0,188.0,Arkadan İtiş,...,0.0,0.0,0.0,0,bilinmiyor,Adana,2025,8,12,1359000


In [233]:
df.to_excel("verisetiTemiz2.xlsx")

In [235]:
df["marka"].describe()

count                                                48258
unique                                                1408
top       Volkswagen_Passat_1.6 TDi BlueMotion Comfortline
freq                                                   532
Name: marka, dtype: object

In [241]:
df["fiyat"].describe()

count    4.825800e+04
mean     7.026173e+05
std      4.655721e+05
min      5.100000e+04
25%      3.550000e+05
50%      5.800000e+05
75%      9.250000e+05
max      2.999000e+06
Name: fiyat, dtype: float64

In [251]:
df[df["fiyat"]<200000].describe()

,kilometre,motor_hacmi,motor_gucu,tramer,boyali_sayisi,degisen_sayisi,tramer_bilinmiyor,ilan_yil,ilan_ay,yas,fiyat
count,3163.000000,3163.000000,3163.000000,3163.000000,3163.000000,3163.000000,3163.000000,3163.0,3163.000000,3163.000000,3163.000000
mean,233742.858678,1513.404995,80.873222,9.470439,1.205185,0.200126,0.416061,2025.0,7.615555,30.418590,157311.346823
std,99139.725454,160.793050,14.092876,206.690761,3.117245,0.622760,0.492982,0.0,0.637366,4.345739,28927.541148
min,6000.000000,1100.000000,50.000000,0.000000,0.000000,0.000000,0.000000,2025.0,2.000000,8.000000,51000.000000
25%,170000.000000,1400.000000,72.000000,0.000000,0.000000,0.000000,0.000000,2025.0,7.000000,28.000000,139000.000000
50%,248000.000000,1500.000000,83.000000,0.000000,0.000000,0.000000,0.000000,2025.0,8.000000,31.000000,160000.000000
75%,300000.000000,1600.000000,88.000000,0.000000,0.000000,0.000000,1.000000,2025.0,8.000000,33.000000,180000.000000
max,500000.000000,2500.000000,170.000000,7900.000000,12.000000,7.000000,1.000000,2025.0,8.000000,52.000000,199999.000000


In [253]:
df[df["fiyat"]<400000].describe()

,kilometre,motor_hacmi,motor_gucu,tramer,boyali_sayisi,degisen_sayisi,tramer_bilinmiyor,ilan_yil,ilan_ay,yas,fiyat
count,14792.000000,14792.000000,14792.000000,14792.000000,14792.000000,14792.000000,14792.000000,14792.0,14792.000000,14792.000000,14792.000000
mean,259555.739657,1497.011898,88.123310,210.945105,2.282991,0.478434,0.289075,2025.0,7.595254,24.569632,272539.615671
std,81963.138549,191.232145,18.371664,2490.598883,3.667391,0.900091,0.453348,0.0,0.650343,5.715060,79810.623283
min,6000.000000,900.000000,50.000000,0.000000,0.000000,0.000000,0.000000,2025.0,1.000000,1.000000,51000.000000
25%,212000.000000,1400.000000,75.000000,0.000000,0.000000,0.000000,0.000000,2025.0,7.000000,20.000000,210000.000000
50%,263000.000000,1500.000000,85.000000,0.000000,0.000000,0.000000,0.000000,2025.0,8.000000,25.000000,280000.000000
75%,310000.000000,1600.000000,100.000000,0.000000,4.000000,1.000000,1.000000,2025.0,8.000000,28.000000,344000.000000
max,500000.000000,3000.000000,225.000000,135026.000000,12.000000,11.000000,1.000000,2025.0,8.000000,52.000000,399999.000000
